# Prognose monatlicher Kfz-Schadenzahlen mit PROC FORECAST

## Zusammenfassung

Ein Reserving-Team im Versicherungswesen benötigt einen 12-Monats-Ausblick auf monatliche Kfz-Schadenzahlen, der ein stetiges Portfolio-Wachstum sowie einen deutlichen Winter-Anstieg zeigt. Dieses Notebook erzeugt fünf Jahre synthetischer monatlicher Schadenzahlen (60 Monate, Jan. 2020 - Dez. 2024, im Bereich von etwa 1.460 bis 2.780 Schäden) und verwendet dann **PROC FORECAST**, um ein schrittweise-autoregressives Basismodell sowie ein multiplikatives Holt-Winters-Saisonmodell anzupassen. Jedes Modell liefert 12 monatliche Punktprognosen mit 95%-Konfidenzgrenzen für die Kapazitäts- und Reserveplanung. Das saisonale Holt-Winters-Modell prognostiziert die stärkste Nachfrage ein bis zwei Monate im Voraus (etwa 2.780-2.900 Schäden) und schwächt sich bis zu einem Tiefpunkt um Schritt neun ab (etwa 2.200), während die autoregressive Basislinie einen glatteren Rückgang prognostiziert; beide Konfidenzbänder verbreitern sich mit dem Horizont.

## Datenquellen

| Datensatz | Zeilen | Granularität | Schlüsselvariablen | Beschreibung |
|---------|------|-------|---------------|-------------|
| `claims` | 60 | Eine Zeile je Kalendermonat (Jan. 2020 - Dez. 2024) | `date` (SAS-Datum, `MONYY7.`), `claim_count` (numerisch) | Synthetische monatliche Kfz-Schadenzahlen, gebildet aus einem linearen Wachstumstrend (~9 Schäden/Monat), einem sinusförmigen Jahreszyklus, einem additiven Dezember/Januar/Februar-Winteranstieg und Gauß'schem Rauschen (`rand('normal')`). Der Seed `20240531` macht es reproduzierbar. |

# Prognose monatlicher Kfz-Schadenzahlen

Reserving- und Kapazitätsplanungsteams bei einem Privatkundenversicherer benötigen einen Vorausblick darauf, wie viele **Kfz-Schäden** jeden Monat gemeldet werden. Das Schadenvolumen in diesem Bestand wächst stetig mit der Ausweitung des Portfolios und steigt jeden Winter sprunghaft an, wenn Eis, Schnee und weniger Tageslicht zu mehr Kollisions- und Glasschäden führen.

Dieses Notebook führt durch einen vollständigen `PROC FORECAST`-Workflow:

1. Erzeugen einer realistischen, vollständig synthetischen monatlichen Schadenzahlenreihe.
2. Anpassen einer **schrittweise-autoregressiven (STEPAR)** Basislinie, die Trend und Autokorrelation erfasst.
3. Anpassen eines **multiplikativen Holt-Winters (WINTERS)**-Modells, das den 12-monatigen Saisonzyklus explizit modelliert.
4. Vergleich der beiden Modelle und Interpretation der Vorwärtsprognose und des Konfidenzbands.

Alles läuft mit synthetischen Inline-Daten - keine externen Dateien oder Netzwerkzugriffe.

## Schritt 1 - Erzeugen der synthetischen Schadenreihe

Der DATA-Step unten erzeugt **60 monatliche Beobachtungen** (Januar 2020 bis Dezember 2024). Für jeden Monat kombinieren wir vier Komponenten, die ein echtes Kfz-Portfolio widerspiegeln:

- **Trend** - eine Basislinie von ~1.800 Schäden, die um ~9 pro Monat wächst, während die Risikoexposition steigt.
- **Jahreszyklus** - ein Sinus-/Kosinus-Term, der eine glatte saisonale Welle erzeugt.
- **Winteranstieg** - ein additiver Anstieg in Dezember, Januar und Februar.
- **Rauschen** - `rand('normal', 0, 90)` für realistische Monat-zu-Monat-Schwankungen.

`call streaminit()` legt den Zufallsstrom fest, damit das Notebook reproduzierbar ist. Die Variable `date` ist ein echtes SAS-Datum, das mit `INTNX` erzeugt und mit `MONYY7.` formatiert wird, und die Anweisung `ID date INTERVAL=MONTH` benennt sie als Zeitkennung. Die ersten 14 unten gedruckten Zeilen liegen zwischen etwa 1.460 und 2.450 Schäden.

In [1]:
DATEN claims;
    AUFRUFEN streaminit(20240531);
    AUSFÜHRUNG i = 0 BIS 59;
        /* Ein echtes SAS-Monatsdatum erzeugen, damit INTERVAL=MONTH passt */
        date = intnx('month', '01JAN2020'd, i);
        format date monyy7.;

        month_idx = mod(i, 12) + 1;          /* 1 = Jan ... 12 = Dez */

        trend  = 1800 + 9 * i;               /* wachsende Risikoexpositionsbasis */
        season = 260 * sin((month_idx - 1) / 12 * 2 * constant('pi'))
               + 150 * cos((month_idx - 1) / 12 * 2 * constant('pi'));
        winter = 220 * (month_idx in (12, 1, 2));   /* Eis-/Schneeanstieg */
        noise  = rand('normal', 0, 90);

        claim_count = round(trend + season + winter + noise);
        WENN claim_count < 0 DANN claim_count = 0;
        AUSGABE;
    ENDE;
    BEHALTEN date claim_count;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=claims(obs=14) BEZEICHNUNG;
    BEZEICHNUNG date = 'Monat' claim_count = 'Gemeldete Schäden';
    TITEL 'Erste 14 Monate synthetischer Kfz-Schadenzahlen';
AUSFÜHREN;

                                    Erste 14 Monate synthetischer Kfz-Schadenzahlen                                     

  Obs  Monat   Gemeldete Schäden
    1  21915                2178
    2  21946                2281
    3  21975                2252
    4  22006                1974
    5  22036                2067
    6  22067                1805
    7  22097                1697
    8  22128                1619
    9  22159                1462
   10  22189                1688
   11  22220                1713
   12  22250                2008
   13  22281                2116
   14  22312                2451

... 46 more observations (showing 14 of 60)




NOTE: DATA claims


NOTE: Wrote claims (60 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=claims

NOTE: PROC PRINT completed: 14 observations printed, 2 variables


## Schritt 2 - Schrittweise-autoregressive Basislinie (METHOD=STEPAR)

`METHOD=STEPAR` ist die Voreinstellung. Es passt zunächst ein Zeittrendmodell an - hier `TREND=2` für einen linearen Trend - und wendet dann eine **schrittweise Autoregression** auf die Residuen an, wobei AR-Lags nach Signifikanz aufgenommen und beibehalten werden. `AR=4` begrenzt die maximale autoregressive Ordnung auf vier Lags, was für eine monatliche Reihe mit kurzfristigem Momentum völlig ausreicht.

Wichtige hier verwendete Optionen:

- `LEAD=12` - Prognose für 12 Monate über die Daten hinaus.
- `ALPHA=0.05` - 95%-Konfidenzgrenzen.
- `OUTFULL` - stapelt die historischen `ACTUAL`-Zeilen zusammen mit den Zeilen des Prognosehorizonts im `OUT=`-Datensatz (statt nur der Prognosezeilen allein).
- `OUTLIMIT` - fügt die Spalten für die untere/obere Konfidenzgrenze hinzu (`L95_claim_count`, `U95_claim_count`).
- `ID date INTERVAL=MONTH` - benennt die Zeitkennung und erklärt die Reihe als monatlich.

In [2]:
PROZEDUR forecast DATEN=claims
        out=fc_stepar
        METHOD=stepar trend=2 ar=4
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VAR claim_count;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=fc_stepar(obs=10) BEZEICHNUNG;
    TITEL 'STEPAR-Ausgabe: Tatsächliche, angepasste und Prognosezeilen';
AUSFÜHREN;

                                    Erste 14 Monate synthetischer Kfz-Schadenzahlen                                     

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                              STEPAR-Ausgabe: Tatsächliche, angepasste und Prognosezeilen                               

  Obs   DATE  _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT
    1  21915  ACTUAL  2121.816667                .                .
    2  21946  ACTUAL  2167.711419                .                .
    3  21975  ACTUAL  2254.781177                .                .
    4  22006  ACTUAL  2228.505912                .                .
    5  22036  ACTUAL  1978.152296                .                .
    6  22067  ACTUAL  2030.606563                .                .
    7  22097  ACTUAL  1864.520418                .                .
    8  22128  ACTUAL  1784.855682                .                .
    9  22159  ACTUAL  1740.781553  


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: PROC PRINT data=fc_stepar

NOTE: PROC PRINT completed: 10 observations printed, 5 variables


### Lesen des OUT=-Datensatzes

Der `OUT=`-Datensatz stapelt Zeilen anhand einer `_TYPE_`-Spalte und führt die Konfidenzgrenzen als zusätzliche **Spalten**:

| Element | Art | Bedeutung |
|---------|------|---------|
| `_TYPE_ = 'ACTUAL'` | Zeile | Der beobachtete `claim_count` für jeden der 60 historischen Monate. |
| `_TYPE_ = 'FORECAST'` | Zeile | Die 12 Punktprognosen für den Prognosehorizont. |
| `L95_claim_count` / `U95_claim_count` | Spalte | Untere / obere 95%-Konfidenzgrenzen, befüllt in den `FORECAST`-Zeilen (fehlend in `ACTUAL`-Zeilen). Das numerische Niveau spiegelt `ALPHA=` wider. |

Der oben gedruckte `OUT=`-Datensatz enthält also 72 Zeilen: 60 `ACTUAL`-Historienzeilen gefolgt von 12 `FORECAST`-Horizontzeilen. Die ersten zehn oben gezeigten Zeilen sind alle `ACTUAL`, wobei die Konfidenzgrenzen-Spalten fehlen, da Grenzen nur an die Prognosezeilen angehängt werden.

> **Engine-Hinweis.** SAS `OUTFULL` schreibt außerdem eine Innerhalb-der-Stichprobe-Einschritt-`FORECAST`-Zeile für jeden historischen Monat, und `OUTRESID` fügt `_TYPE_='RESIDUAL'`-Zeilen hinzu. Jenners PROC FORECAST gibt diese In-Sample-Fitted-/Residuenzeilen noch nicht aus (verfolgt als Gap-Test `400979`), daher liest dieses Notebook nur die `ACTUAL`-Historie und den vorwärtsgerichteten `FORECAST`-Horizont.

## Schritt 3 - Saisonales Holt-Winters-Modell (METHOD=WINTERS)

Das STEPAR-Modell erfasst Trend und kurzfristige Autokorrelation, modelliert aber nicht explizit den wiederkehrenden Dezember-Februar-Anstieg. Für eine Reihe mit einem klaren Jahresrhythmus ist **multiplikatives Holt-Winters** das bessere Werkzeug.

`METHOD=WINTERS` zerlegt die Reihe in Niveau, linearen Trend (`TREND=2`) und einen **multiplikativen Saisonfaktor**. `SEASONS=12` erklärt einen 12-periodigen (monatlichen) Saisonzyklus. Wir fordern erneut einen 12-Monats-`LEAD` mit 95%-Grenzen sowie `OUTFULL OUTLIMIT` an, damit die Ausgabe mit dem STEPAR-Lauf übereinstimmt.

Da die Engine die Prognose-`ID` um eine Einheit pro Schritt vorrückt (statt `INTERVAL=MONTH` für die Horizontdaten zu berücksichtigen - Gap-Test `400979`), betrachtet die Zelle unten die Prognose anhand von **Monaten im Voraus (Schritt 1-12)**, statt sich auf die gedruckten Kalenderdaten zu verlassen.

In [3]:
PROZEDUR forecast DATEN=claims
        out=fc_winters
        METHOD=winters seasons=12 trend=2
        LEAD=12 ALPHA=0.05
        outfull outlimit;
    id date interval=month;
    VAR claim_count;
AUSFÜHREN;

/* Den 12-monatigen Vorwärtshorizont behalten und nach Schritt indizieren (1..12). */
DATEN horizon;
    FESTLEGEN fc_winters;
    BEHALTEN_W months_ahead 0;
    WENN _type_ = 'FORECAST';
    months_ahead + 1;
    BEHALTEN months_ahead claim_count l95_claim_count u95_claim_count;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=horizon BEZEICHNUNG;
    BEZEICHNUNG months_ahead     = 'Monate voraus'
          claim_count       = 'Prognostizierte Schäden'
          l95_claim_count   = 'Untere 95%-Grenze'
          u95_claim_count   = 'Obere 95%-Grenze';
    TITEL 'Holt-Winters 12-Monats-Vorwärtsprognose (nach Schritt)';
AUSFÜHREN;

                              STEPAR-Ausgabe: Tatsächliche, angepasste und Prognosezeilen                               

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 60
Forecast Periods: 12



                                 Holt-Winters 12-Monats-Vorwärtsprognose (nach Schritt)                                 

  Obs  Monate voraus   Prognostizierte Schäden  Untere 95%-Grenze  Obere 95%-Grenze
    1              1                2783.07951        2577.844742       2988.314278
    2              2               2897.355557        2607.109764       3187.601349
    3              3               2805.272075        2449.795029        3160.74912
    4              4               2664.498049        2254.028514       3074.967585
    5              5               2628.810136        2169.891244       3087.729029
    6              6                2548.39319        2045.672732       3051.113649
    7              7               2391.649756          184


NOTE: PROC FORECAST data=claims

NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 72 observations.
NOTE: DATA horizon


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote horizon (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=horizon

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Schritt 4 - Die beiden Modelle direkt vergleichen

Der klarste Weg zu beurteilen, ob sich das saisonale Modell lohnt, besteht darin, seine 12-Monats-Prognose Schritt für Schritt neben die STEPAR-Basislinie zu legen. Wir ziehen die `FORECAST`-Zeilen aus beiden `OUT=`-Datensätzen, indizieren jede nach Monaten im Voraus und führen sie zusammen, sodass die Abweichung auf einen Blick sichtbar wird.

Die beiden Methoden stimmen im groben Niveau überein, sind sich aber in der **Form** uneinig: Holt-Winters trägt den Jahresrhythmus in den Horizont (ein höheres Niveau im frühen Horizont, das zu einem Tiefpunkt in der Mitte des Horizonts abklingt und dann wieder ansteigt), während STEPAR - das Saisonalität nur indirekt über AR-Lags modelliert - glatter zum Reihenmittelwert hin abklingt. Die Lücke zwischen ihnen bei jedem Schritt ist das saisonale Signal, das STEPAR liegen lässt.

> SAS würde diese Angemessenheitsprüfung normalerweise mit `OUTRESID`-Einschritt-Residuen (`_TYPE_='RESIDUAL'`) durchführen. Jenner befüllt diese Zeilen noch nicht (Gap-Test `400979`), daher vergleichen wir stattdessen die beiden Vorwärtsprognosen direkt - eine Diagnose, die nur Ausgaben verwendet, die die Engine tatsächlich erzeugt.

In [4]:
/* STEPAR-Horizont, indiziert nach Monaten im Voraus. */
DATEN stepar_h;
    FESTLEGEN fc_stepar;
    BEHALTEN_W months_ahead 0;
    WENN _type_ = 'FORECAST';
    months_ahead + 1;
    stepar = claim_count;
    BEHALTEN months_ahead stepar;
AUSFÜHREN;

/* WINTERS-Horizont, indiziert nach Monaten im Voraus. */
DATEN winters_h;
    FESTLEGEN fc_winters;
    BEHALTEN_W months_ahead 0;
    WENN _type_ = 'FORECAST';
    months_ahead + 1;
    winters = claim_count;
    BEHALTEN months_ahead winters;
AUSFÜHREN;

/* Beide zusammenführen und die saisonale Lücke zeigen, die STEPAR übersieht. */
DATEN COMPARE;
    ZUSAMMENFÜHREN stepar_h winters_h;
    NACH months_ahead;
    seasonal_gap = winters - stepar;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=COMPARE BEZEICHNUNG;
    BEZEICHNUNG months_ahead = 'Monate voraus'
          stepar        = 'STEPAR-Prognose'
          winters       = 'Winters-Prognose'
          seasonal_gap  = 'Winters - STEPAR';
    format stepar winters seasonal_gap 8.0;
    TITEL 'STEPAR vs. Holt-Winters: Vergleich der 12-Monats-Prognose';
AUSFÜHREN;

                               STEPAR vs. Holt-Winters: Vergleich der 12-Monats-Prognose                                

  Obs  Monate voraus  STEPAR-Prognose  Winters-Prognose  Winters - STEPAR
    1              1             2619              2783               164
    2              2             2537              2897               361
    3              3             2384              2805               421
    4              4             2214              2664               450
    5              5             2089              2629               540
    6              6             2010              2548               539
    7              7             1982              2392               410
    8              8             1995              2240               245
    9              9             2031              2197               166
   10             10             2075              2354               280
   11             11             2115              2423         


NOTE: DATA stepar_h


NOTE: Read 72 rows from fc_stepar.
NOTE: Wrote stepar_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA winters_h


NOTE: Read 72 rows from fc_winters.
NOTE: Wrote winters_h (12 rows, 2 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA compare

NOTE: Stream 1 processed 12 rows, max BY-group size: 1 (O(1) memory verified)
NOTE: Stream 2 processed 12 rows, max BY-group size: 1 (O(1) memory verified)

NOTE: Wrote compare (12 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=compare

NOTE: PROC PRINT completed: 12 observations printed, 4 variables


## Schritt 5 - Jede Sparte auf einmal prognostizieren (BY-Verarbeitung)

Reale Reserving-Läufe umfassen mehrere Produkte. Bei nach `line_of_business` sortierten Daten lässt eine `BY`-Anweisung `PROC FORECAST` ein **unabhängiges saisonales Modell für jede Gruppe** in einem einzigen Aufruf anpassen. Hier synthetisieren wir eine Kfz-Sparte (höheres Basisvolumen) und eine Wohngebäude-Sparte (niedrigeres Basisvolumen) und prognostizieren beide sechs Monate im Voraus. `OUTALL` schreibt den vollständigen Satz an Zeilen - die `ACTUAL`-Historie und den `FORECAST`-Horizont - zusammen mit den Grenzspalten für jede Gruppe, und wir behalten nur die `FORECAST`-Zeilen für die Tabelle unten.

In [5]:
DATEN multi_book;
    AUFRUFEN streaminit(20240531);
    LÄNGE line_of_business $12;
    AUSFÜHRUNG lob = 1 BIS 2;
        WENN lob = 1 DANN line_of_business = 'Kfz';
        SONST            line_of_business = 'Wohngebäude';
        AUSFÜHRUNG i = 0 BIS 47;
            date = intnx('month', '01JAN2021'd, i);
            format date monyy7.;
            mi   = mod(i, 12) + 1;
            BASE = (lob = 1) * 2000 + (lob = 2) * 1400;
            claim_count = round(BASE + 8 * i
                + 200 * sin((mi - 1) / 12 * 2 * constant('pi'))
                + 180 * (mi in (12, 1, 2))
                + rand('normal', 0, 70));
            AUSGABE;
        ENDE;
    ENDE;
    BEHALTEN line_of_business date claim_count;
AUSFÜHREN;

PROZEDUR SORTIEREN DATEN=multi_book;
    NACH line_of_business date;
AUSFÜHREN;

PROZEDUR forecast DATEN=multi_book
        out=fc_by
        METHOD=winters seasons=12 trend=2
        LEAD=6 ALPHA=0.05
        outall;
    NACH line_of_business;
    id date interval=month;
    VAR claim_count;
AUSFÜHREN;

PROZEDUR DRUCKEN DATEN=fc_by(obs=12) BEZEICHNUNG;
    WO _type_ = 'FORECAST';
    TITEL 'Prognosen je Sparte über 6 Monate';
AUSFÜHREN;

                               STEPAR vs. Holt-Winters: Vergleich der 12-Monats-Prognose                                


BY Group: line_of_business=Kfz

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6


BY Group: line_of_business=Wohngebäude

The FORECAST Procedure

Variables Forecasted: claim_count
Number of Observations: 48
Forecast Periods: 6



                                           Prognosen je Sparte über 6 Monate                                            

  Obs  LINE_OF_BUSINESS   DATE    _TYPE_  CLAIM_COUNT  L95_CLAIM_COUNT  U95_CLAIM_COUNT  RESIDUAL_CLAIM_COUNT
    1  Kfz               23742  FORECAST  2524.596717      2359.095846      2690.097589                     .
    2  Kfz               23773  FORECAST  2604.418724      2370.365147        2838.4723                     .
    3  Kfz               23801  FORECAST  2576.092313      2289.436395       2862.74823                     .
    4  Kfz               238


NOTE: DATA multi_book


NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds
NOTE: PROC SORT data=multi_book

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 96 rows from multi_book.
NOTE: Wrote multi_book (96 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC FORECAST data=multi_book

NOTE: BY Group: line_of_business=Kfz
NOTE: Using Python for FORECAST estimation
NOTE: BY Group: line_of_business=Wohngebäude
NOTE: Using Python for FORECAST estimation
NOTE: Output dataset created with 108 observations.
NOTE: PROC PRINT data=fc_by

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: PROC PRINT completed: 6 observations printed, 7 variables


## Interpretation der Ergebnisse

**Was die Modelle dem Reserving-Team sagen:**

- **STEPAR** reproduziert den Aufwärtstrend und das kurzfristige Momentum, aber seine 12-Monats-Prognose klingt glatt von etwa 2.620 (Schritt 1) auf rund 1.980 in der Mitte des Horizonts ab, bevor sie auf etwa 2.140 zurückdriftet - es reproduziert keinen scharfen Winterhöhepunkt, da es Saisonalität nur über autoregressive Lags berücksichtigt. Es ist eine vernünftige schnelle Basislinie.
- **WINTERS** mit `SEASONS=12` trägt den Jahresrhythmus direkt über seine multiplikativen Saisonfaktoren: Seine Prognose ist im frühen Horizont am höchsten (etwa 2.780 bei Schritt 1, etwa 2.900 bei Schritt 2), schwächt sich zu einem Tiefpunkt nahe Schritt 9 ab (etwa 2.200) und steigt bis Schritt 12 wieder an (etwa 2.800). Gegenüber der STEPAR-Basislinie liegt sie über den größten Teil des Horizonts **150-660 Schäden höher** (siehe die Spalte `Winters - STEPAR` in Schritt 4) - diese Lücke ist das saisonale Signal, das das autoregressive Modell auslässt.
- Das **95%-Konfidenzband** (Spalten `L95_*`/`U95_*`, gesteuert durch `ALPHA=`) verbreitert sich mit zunehmendem Horizont - beim WINTERS-Modell von einer Breite von etwa 410 Schäden bei Schritt 1 auf etwa 1.420 bei Schritt 12 - das ehrliche Signal, dass Schätzungen am Ende des Horizonts mehr Unsicherheit tragen als kurzfristige. Reserving-Verantwortliche sollten Kapital gegen die obere Grenze vorhalten, nicht nur gegen die Punktprognose.
- Die **BY-Verarbeitung** erzeugt die Kfz- und Wohngebäude-Prognosen in einem Durchgang, jede mit ihrer eigenen saisonalen Anpassung. Die Kfz-Sparte prognostiziert im Bereich von etwa 2.510-2.600, während die Wohngebäude-Sparte bei etwa 1.870-2.030 liegt, sodass jede Sparte ihr eigenes Niveau und ihre eigene saisonale Form behält - das Muster, das das Team auf jedes Produkt im Portfolio ausdehnen würde.

**Fazit:** Für eine Schadenzahlenreihe mit einem klaren Jahreszyklus ist `METHOD=WINTERS SEASONS=12` die naheliegende Wahl; die STEPAR-Basislinie ist eine nützliche Plausibilitätsprüfung, und `OUTFULL`/`OUTLIMIT` zusammen mit einem schrittweisen Modellvergleich erlauben es dem Aktuar, die Vorwärtsreserve mit ihrem Unsicherheitsband zu bemessen.

> **Engine-Fidelity-Hinweis.** Dieses Notebook dokumentiert zwei aktuelle Jenner-PROC-FORECAST-Einschränkungen (Gap-Test `400979`): Die Prognosehorizont-`ID` wird um eine Einheit pro Schritt vorgerückt statt um `INTERVAL=MONTH`, sodass die gedruckten Prognosedaten nicht die beabsichtigten Kalendermonate des Jahres 2025 sind (wir betrachten den Horizont stattdessen nach Schritt); und `OUTRESID`/`OUTALL` befüllen die Einschritt-`_TYPE_='RESIDUAL'`-Zeilen noch nicht, sodass Residuendiagnosen durch einen direkten Zweimodellvergleich ersetzt werden. Die Punktprognosen und Konfidenzgrenzen selbst werden erzeugt und sind das, worauf die obige Erzählung basiert.